# reparameterization-trick — worked example 1: Sample from a Gaussian Using the Reparameterization Trick

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `reparameterization-trick`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

To train a VAE, we need gradients to flow through a sampling step `z ~ N(mu, sigma^2)`. Sampling is non-differentiable, but the reparameterization trick fixes this: draw noise `eps ~ N(0, 1)` independently (no grad), then compute `z = mu + sigma * eps`. Now `z` is a deterministic function of `(mu, sigma)`, so backpropagation works normally. The encoder is parameterized as `(mu, logsigma)` where `sigma = exp(0.5 * logsigma)`.

## Worked solution

**Step 1 — understand why we use `logsigma`.** The log-variance (or half-log-variance) parameterization ensures `sigma > 0` without a constraint. `sigma = exp(0.5 * logsigma)` because `logsigma = log(sigma^2)`, so `0.5 * logsigma = log(sigma)`, and `exp(log(sigma)) = sigma`.

**Step 2 — draw `eps`.** Use `t.randn_like(mu)` to draw a noise tensor with the same shape, dtype, and device as `mu`. This is better than `t.randn(*mu.shape)` because it handles GPU tensors automatically.

**Step 3 — compute `z`.** `z = mu + sigma * eps`. This shifts the unit normal by `mu` and scales by `sigma`, producing a sample from `N(mu, sigma^2)`.

**Step 4 — check gradient flow.** Because `eps` is detached from the computation graph, PyTorch can differentiate `z` with respect to `mu` (gradient = 1) and `logsigma` (gradient = `0.5 * sigma * eps`).

**Step 5 — verify.** Over many samples, the mean of `z` should approach `mu` and the std should approach `sigma`.

In [ ]:
import torch as t

def reparameterize(mu: t.Tensor, logsigma: t.Tensor) -> t.Tensor:
    """Draw a differentiable sample z ~ N(mu, exp(logsigma))."""
    sigma = (0.5 * logsigma).exp()   # sigma = exp(logsigma/2)
    eps = t.randn_like(mu)            # eps ~ N(0, 1), no grad
    return mu + sigma * eps

# --- exercise and print ---
t.manual_seed(0)
B, L = 4, 3

mu = t.tensor([[1.0, 0.0, -1.0]] * B)      # fixed mean
logsigma = t.tensor([[0.0, 0.0, 0.0]] * B) # logsigma=0 => sigma=1

t.manual_seed(17)
samples = t.stack([reparameterize(mu, logsigma) for _ in range(1000)])
print('True mu:       ', mu[0].tolist())
print('Sample mean:   ', samples[:, 0, :].mean(0).tolist())   # approx mu
print('True sigma:    ', (0.5 * logsigma[0]).exp().tolist())  # [1,1,1]
print('Sample std:    ', samples[:, 0, :].std(0).tolist())    # approx 1
print('Output shape:  ', reparameterize(mu, logsigma).shape)  # (4, 3)